# 기준 획 모델 V2
GPU 런타임을 선택하세요. 학습된 모델이 아니라 새 모델의 학습 노트북입니다.
첫 셀이 실제 GitHub 저장소에서 코드를 자동으로 받습니다. ZIP 업로드는 필요 없습니다. 두 번째 셀이 기존 paths.csv를 자동 변환합니다. 기준/정답 획은 CSV에서 같은 순서여야 합니다. 데이터 형식은 REFERENCE_V2.md를 참조하세요. 기존 best.pt와는 호환되지 않습니다.


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile
root = Path(tempfile.mkdtemp(prefix="reference_v2_", dir="/content"))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/yechan25/hsqm.git", str(root)], check=True)
if not (root / "requirements.txt").is_file():
    raise FileNotFoundError("저장소에서 requirements.txt를 찾지 못했습니다.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key == "stroke_model" or key.startswith("stroke_model."):
        del sys.modules[key]
commit = subprocess.check_output(["git", "-C", str(root), "rev-parse", "--short", "HEAD"], text=True).strip()
print("모델 코드 준비 완료 / GitHub 버전:", commit)


In [ ]:
from google.colab import drive
from datetime import datetime
drive.mount("/content/drive")
from stroke_model.convert_reference_csv import convert_csv, existing_path
from stroke_model.preview_reference import preview_reference
csv_path = existing_path("/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/train_dataset/paths.csv")
# 원본 CSV/이미지는 변경하지 않습니다. train CSV에서만 80/20으로 나눕니다.
prepared = csv_path.parents[2] / "reference_v2_data" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
manifest = convert_csv(csv_path, prepared)
preview_reference(manifest)
print("데이터 준비 완료:", manifest)


In [ ]:
# 학습 데이터만 증강합니다. 0으로 바꾸면 증강 없이 비교할 수 있습니다.
variants = 20
training_manifest = manifest
if variants > 0:
    # 셀을 다시 실행해도 기존 증강 폴더를 덮어쓰지 않습니다.
    augment_seed = 42
    while (manifest.parent / f"augmented_seed{augment_seed}").exists():
        augment_seed += 1
    subprocess.run([sys.executable, "-m", "stroke_model.augment_reference",
                    "--manifest", str(manifest), "--variants", str(variants),
                    "--seed", str(augment_seed)], cwd=root, check=True)
    training_manifest = manifest.parent / f"augmented_seed{augment_seed}" / "manifest.json"
print("학습 manifest:", training_manifest)


In [ ]:
from datetime import datetime
epochs = 30
output = manifest.parent / ("reference_v2_run_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
subprocess.run([sys.executable, "-m", "stroke_model.train_reference",
                "--manifest", str(training_manifest), "--epochs", str(epochs),
                "--output", str(output)], cwd=root, check=True)
print("체크포인트:", output / "best.pt")


In [ ]:
# 학습이 끝난 후 가장 좋은 모델의 검증 이미지 결과를 확인합니다.
from stroke_model.preview_reference import preview_reference
preview_reference(manifest, checkpoint=output / "best.pt", count=5)
